1. Initial Inspection

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_PATH = '../data/synthetic_orders_ctgan.csv' if os.path.exists('../data/synthetic_orders_ctgan.csv') else 'data/synthetic_orders_ctgan.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,order_id,customer_id,store_id,order_timestamp,delivery_timestamp,order_value
0,sdv-id-PPcBpb,C018496,DS007,2026-05-29 11:30:12,2026-05-29 11:40:14.121021603,204.0
1,sdv-id-NdEuVy,C006527,DS015,2026-07-15 00:01:52,2026-07-15 00:06:52.251103467,710.0
2,sdv-id-kPbvnT,C008719,DS007,2026-07-23 10:39:54,2026-07-23 10:49:54.706517132,251.0
3,sdv-id-JrFXLd,C013876,DS002,2026-06-02 02:06:09,2026-06-02 02:12:09.572774151,529.0
4,sdv-id-rNMsjf,C011488,DS005,2026-07-03 07:43:44,2026-07-03 07:49:44.910442625,515.0


In [50]:
df.sample(20)

,order_id,customer_id,store_id,order_timestamp,delivery_timestamp,order_value
449328,sdv-id-VyGjnB,C014505,DS004,2026-05-24 01:11:06,2026-05-24 01:17:06.782697167,107.0
47992,sdv-id-pMuBhb,C007837,DS008,2026-06-22 11:41:19,2026-06-22 11:51:20.624869948,293.0
183209,sdv-id-wbHuPi,C011541,DS008,2026-06-02 22:04:02,2026-06-02 22:15:02.000000000,131.0
351959,sdv-id-ldqdfV,C019790,DS007,2026-07-18 11:24:14,2026-07-18 11:30:14.253378612,539.0
281377,sdv-id-UHrIeH,C001342,DS011,2026-07-22 04:14:02,2026-07-22 04:25:00.954966836,160.0
472816,sdv-id-MZXaOo,C013269,DS006,2026-04-25 18:49:16,2026-04-25 18:58:16.419613508,314.0
23487,sdv-id-qBjJts,C016409,DS011,2026-05-07 16:55:46,2026-05-07 17:05:46.835330044,736.0
338724,sdv-id-MyUTTi,C010866,DS008,2026-07-13 02:47:11,2026-07-13 02:54:11.204153275,131.0
318407,sdv-id-JqHwqm,C001457,DS012,2026-06-19 14:30:53,2026-06-19 14:34:53.431673854,803.0
1391,sdv-id-hjXRtg,C008764,DS010,2026-06-20 14:41:38,2026-06-20 14:46:37.815619557,1434.0


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495874 entries, 0 to 495873
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order_id            495874 non-null  object 
 1   customer_id         495874 non-null  object 
 2   store_id            495874 non-null  object 
 3   order_timestamp     495874 non-null  object 
 4   delivery_timestamp  495874 non-null  object 
 5   order_value         495874 non-null  float64
dtypes: float64(1), object(5)
memory usage: 22.7+ MB


In [52]:
df.shape

(495874, 6)

In [53]:
df.isnull().sum()

order_id              0
customer_id           0
store_id              0
order_timestamp       0
delivery_timestamp    0
order_value           0
dtype: int64

In [54]:
df.isna().sum()

order_id              0
customer_id           0
store_id              0
order_timestamp       0
delivery_timestamp    0
order_value           0
dtype: int64

In [55]:
df.duplicated().sum()

np.int64(0)

2. Data Cleaning

In [56]:
# 1. Convert to Datetime
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])
df['delivery_timestamp'] = pd.to_datetime(df['delivery_timestamp'])

In [57]:
# 2. Sanity Check: Ensure no delivery happened before an order
invalid_deliveries = (df['delivery_timestamp'] < df['order_timestamp']).sum()
print(f"Invalid Deliveries Found: {invalid_deliveries}")

Invalid Deliveries Found: 0


In [58]:
# 3. trimming the timestamps for delivery_timestamp
df['delivery_timestamp'] = df['delivery_timestamp'].astype(str).str[:19]

In [59]:
# 4. Remove invalid deliveries
df = df[df['delivery_timestamp'] >= df['order_timestamp']]


In [60]:
# final check
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495874 entries, 0 to 495873
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   order_id            495874 non-null  object        
 1   customer_id         495874 non-null  object        
 2   store_id            495874 non-null  object        
 3   order_timestamp     495874 non-null  datetime64[ns]
 4   delivery_timestamp  495874 non-null  object        
 5   order_value         495874 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 22.7+ MB


In [61]:
df.sample(5)

,order_id,customer_id,store_id,order_timestamp,delivery_timestamp,order_value
175257,sdv-id-OxeJKG,C011042,DS011,2026-07-17 09:24:54,2026-07-17 09:35:54,389.0
389845,sdv-id-zlfuCc,C016937,DS008,2026-03-27 11:50:00,2026-03-27 12:01:00,258.0
88039,sdv-id-Ywrcyi,C003754,DS002,2026-06-06 05:13:08,2026-06-06 05:20:09,189.0
289450,sdv-id-CcHIWd,C001130,DS006,2026-03-26 16:12:22,2026-03-26 16:16:22,670.0
192306,sdv-id-xCfhIp,C002611,DS006,2026-06-21 21:37:47,2026-06-21 21:41:48,2180.0


In [62]:
# saving the cleaned data to a new CSV file
OUT_PATH = '../data/cleaned_dataset.csv' if os.path.exists('../data') else 'data/cleaned_dataset.csv'
df.to_csv(OUT_PATH, index=False)